# Chapter 9: Evaluation and Deployment
A VLA that is accurate but slow is not a working robot policy. This chapter covers the two things that decide whether a model survives contact with hardware: **honest closed-loop evaluation**, and **inference latency**.

The core of this chapter is numpy-only -- it runs anywhere, no GPU, no simulator.

In [ ]:
!pip install numpy matplotlib

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/09_eval_and_deploy

## The Problem

A robot arm runs its control loop at 30 Hz -- a new action every **33 ms**. One forward pass of our Chapter 7 SmolVLA takes **73 ms** on an RTX 4090 (SigLIP + SmolLM2 prefix + 10 Euler steps).

The policy is more than twice too slow to be called every step. No amount of engineering fixes that by making the model faster; the fix is to stop calling it every step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CONTROL_HZ = 30
control_period_ms = 1000 / CONTROL_HZ
policy_latency_ms = 72.6      # measured, Ch07 SmolVLA on RTX 4090

print(f"Control period:  {control_period_ms:.1f} ms  ({CONTROL_HZ} Hz)")
print(f"Policy latency:  {policy_latency_ms:.1f} ms")
print(f"Ratio:           {policy_latency_ms / control_period_ms:.1f}x too slow")

## Action Chunking

The model already predicts **K future actions at once** -- that is what Chapters 4 and 6 built. So call it once, then execute the chunk one action per control step. One inference now covers K steps.

`ActionChunkBuffer` is just a FIFO of single actions popped from those chunks.

In [ ]:
from async_inference import ActionChunkBuffer, MockPolicy

policy = MockPolicy(action_dim=6, chunk_size=10)
buf = ActionChunkBuffer()

obs = np.zeros(6, dtype=np.float32)
buf.push_chunk(policy.predict_chunk(obs))
print(f"After one inference, buffered actions: {len(buf)}")

for i in range(3):
    print(f"  pop {i}: {np.round(buf.pop(), 3)}")
print(f"Remaining: {len(buf)}")

## Sync vs Async

**Sync**: run the buffer dry, then block for 73 ms while the robot waits. The arm stutters every K steps.

**Async**: fire the next inference on a background thread *before* the buffer empties, so it completes while the robot is still executing buffered actions. Same actions, no stall.

Below, the "robot" sleeps `control_period` each step and the "policy" sleeps its latency.

In [ ]:
import time
from async_inference import SyncChunkController, AsyncChunkController, run_rollout

N_STEPS = 60
CONTROL_PERIOD_S = 0.005     # scaled down so the demo runs fast
LATENCY_S = 0.011            # same 2.2x ratio as 33 ms vs 73 ms

def observe(i):
    return np.full(6, float(i), dtype=np.float32)

def step_env(action):
    time.sleep(CONTROL_PERIOD_S)

ideal_ms = N_STEPS * CONTROL_PERIOD_S * 1000
runs = {}
for label, make in [("Sync", SyncChunkController),
                    ("Async", AsyncChunkController)]:
    p = MockPolicy(action_dim=6, chunk_size=10, latency_s=LATENCY_S)
    ctrl = make(p)
    res = run_rollout(ctrl, observe, step_env, n_steps=N_STEPS)
    ctrl.close()
    runs[label] = res
    overhead = (res.wall_time_s * 1000 - ideal_ms) / ideal_ms * 100
    print(f"{label:<6} wall={res.wall_time_s*1000:7.1f} ms   "
          f"overhead=+{overhead:5.1f}%   policy calls={res.n_inferences}")

print(f"\nIdeal (zero-latency policy): {ideal_ms:.1f} ms")

## Does Async Change What the Robot Does?

The guarantee is precise, and worth stating exactly: **fed the same observation, the async controller emits exactly the actions the sync one would.** The controller logic is identical; only the moment of inference moves.

But in a real rollout the observation is *not* the same, and that is the whole point. Async replans a few steps early, so it queries the policy on a **fresher** observation than sync would have used. The actions therefore differ slightly -- not from a bug, but because async is acting on newer information.

Both cases are shown below.

In [ ]:
# Case 1: constant observation -- the controllers must agree exactly.
obs_const = np.full(6, 2.0, dtype=np.float32)

s = SyncChunkController(MockPolicy(action_dim=6, chunk_size=10))
sync_const = [s.get_action(obs_const) for _ in range(20)]; s.close()

a = AsyncChunkController(MockPolicy(action_dim=6, chunk_size=10), replan_threshold=3)
async_const = [a.get_action(obs_const) for _ in range(20)]; a.close()

print("Constant observation:")
print(f"  identical: {np.allclose(np.stack(sync_const), np.stack(async_const))}")

# Case 2: the live rollout above, where the observation advances every step.
sync_actions = np.stack(runs["Sync"].actions)
async_actions = np.stack(runs["Async"].actions)
diff = np.abs(sync_actions - async_actions)

print("\nLive observation stream:")
print(f"  identical: {np.allclose(sync_actions, async_actions)}")
print(f"  max difference: {diff.max():.2e}  (async replanned on a fresher observation)")
print(f"  steps that differ: {int((diff.max(axis=1) > 0).sum())} / {len(diff)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(sync_actions[:, 0], label="Sync", lw=3, alpha=0.5)
axes[0].plot(async_actions[:, 0], label="Async", lw=1.2, ls="--")
axes[0].set_title("Executed action (dim 0)")
axes[0].set_xlabel("Control step"); axes[0].legend()

axes[1].plot(diff.max(axis=1), color="tab:red")
axes[1].set_title("Per-step max |sync - async|")
axes[1].set_xlabel("Control step")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Temporal Ensembling

Chunks overlap: at step *t* you may hold predictions for *t* made at several earlier timesteps. ACT averages them with exponentially higher weight on the **freshest** prediction, which smooths the seam where one chunk hands off to the next.

In [ ]:
from async_inference import TemporalEnsembler

ens = TemporalEnsembler(m=0.1)
ens.add_chunk(0, np.ones((4, 2)) * 1.0)     # predicts t=0..3
ens.add_chunk(2, np.ones((4, 2)) * 2.0)     # predicts t=2..5, fresher

for t in range(4):
    a = ens.step()
    n = "1 prediction" if t < 2 else "2 predictions (averaged, fresher weighted higher)"
    print(f"t={t}: action={np.round(a, 4)}   <- {n}")

## Evaluating a Policy

`MockManipulationEnv` is a dependency-free reaching task, so the eval harness is exercisable without installing a simulator. The metrics are the ones that matter for manipulation: **success rate**, episode length, and action **smoothness** (jerky policies wear out hardware).

In [ ]:
from eval_libero import MockManipulationEnv, ScriptedReachPolicy, evaluate_policy

policy = ScriptedReachPolicy(action_dim=6, chunk_size=10)

for use_async in (False, True):
    m = evaluate_policy(policy, lambda: MockManipulationEnv(),
                        n_episodes=20, use_async=use_async)
    mode = "async" if use_async else "sync"
    print(f"{mode:<6} success={m['success_rate']*100:5.1f}%  "
          f"length={m['mean_length']:5.1f}  smoothness={m['mean_smoothness']:.4f}  "
          f"(n={int(m['n_episodes'])})")

## The Real Benchmark: LIBERO

`MockManipulationEnv` proves the harness works. **LIBERO** is the benchmark the field actually reports on -- 4 suites probing different kinds of generalization.

It is not on PyPI and pulls in a MuJoCo simulator, so it is gated: `make_libero_env` raises with install guidance rather than failing obscurely.

In [ ]:
from eval_libero import describe_libero_suites, make_libero_env

print(describe_libero_suites())

try:
    env = make_libero_env("libero_spatial", task_id=0)
    print("\nLIBERO available -- running the real benchmark.")
except Exception as e:
    print(f"\nLIBERO not installed (expected):\n  {e}")

## Real Measured Results

The numbers above use a mock policy so the notebook runs anywhere. `benchmark_latency.py` runs the same controllers against the **actual Chapter 7 SmolVLA** (`uv sync --extra vla`). On an RTX 4090, 60 steps at 30 Hz:

| | Raw latency | Rollout wall | Overhead | Policy calls |
|---|---|---|---|---|
| ideal | -- | 1980 ms | -- | -- |
| Sync | 72.6 +/- 1.8 ms | 2432 ms | **+22.8%** | 6 |
| Async | (hidden) | 2081 ms | **+5.1%** | 7 |

Async pays the 73 ms only once, on the cold-start chunk. Sync pays it every 10 steps.

## What We Learned

**Latency is an architecture problem, not an optimization problem.** A 73 ms policy cannot drive a 33 ms control loop by being tuned. Action chunking changes the ratio: one inference now covers 10 steps, so the *effective* requirement becomes 330 ms.

**Async inference is nearly free** -- +5.1% overhead versus +22.8% for sync. The only blocking wait is the cold-start chunk. It is not bit-identical to sync in a live rollout, and should not be: it replans on a fresher observation, which is a feature.

**Report closed-loop success, not loss.** Chapter 5 is the cautionary tale: MAE 0.05 and 0% success. If a VLA result does not come from rollouts, it does not mean much.

**Where this goes next** -- and we did not build these: RL fine-tuning past the limits of behavior cloning, sim-to-real transfer, and multi-embodiment models that share one policy across robot morphologies.

That is the arc: a 3-layer CNN on a toy pushing task, to a SmolVLA-like model with a frozen SigLIP encoder, a frozen SmolLM2 backbone, a flow-matching action expert, LoRA adaptation, and a real-time async controller. Every piece built from scratch, and every result measured rather than assumed.